<a href="https://colab.research.google.com/github/diegozuu/asistente-para-planificacion-de-toma-de-ramos/blob/main/PROTOTIPO%203%20EVAUACION%202/PROTOTIPO_3_EVALUACION_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS


In [ ]:
!apt-get install -y tesseract-ocr tesseract-ocr-spa -q
!pip install gradio openai pandas pytesseract Pillow opencv-python-headless -q

import re
import json
import difflib
import unicodedata
import numpy as np
import cv2
import pandas as pd
import gradio as gr
from openai import OpenAI
from google.colab import userdata
import pytesseract
from PIL import Image

print("[Exito] Librerias importadas correctamente con soporte para imagenes (OCR + vision)")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (5.3.4-1build5).
The following NEW packages will be installed:
  tesseract-ocr-spa
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 1,065 kB of archives.
After this operation, 2,309 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 tesseract-ocr-spa all 1:4.1.0-2 [1,065 kB]
Fetched 1,065 kB in 1s (1,331 kB/s)
Selecting previously unselected package tesseract-ocr-spa.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-spa_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-spa (1:4.1.0-2) ...
Setting up tesseract-ocr-spa (1:4.1.0-2) ...
[Exito] Librerias importadas correctamente con soporte para imagenes (OCR + vision)


# 2. CONEXIÓN A LA API Y CARGA DEL EXCEL

**¿Qué hace esta celda?**
Establece dos conexiones: con el servicio de inteligencia artificial y con el archivo Excel que contiene los horarios.

**Parte 1: conexión con la inteligencia artificial**
- Se usa el servicio **Groq** con el modelo `openai/gpt-oss-120b`.
- La clave de acceso no está escrita en el código: se toma de los Secretos de Colab (`OPENAI_API_KEY`), para que no quede expuesta si se comparte el cuaderno.
- Si la clave falta o es incorrecta, se muestra un mensaje `[Error]` y el resto del cuaderno no podrá usar la IA.

**Parte 2: carga del Excel**
- Hoja `BASE`: contiene todas las secciones publicadas, con su carrera, plan, jornada, docente y horario. Se guarda en `df_horarios`.
- Hoja `OPTATIVOS COMPETENCIAS GLOBALES`: contiene las opciones para el cupo de Competencias Globales. Se guarda en `df_competencias` y se le cambian dos nombres de columna (`Sigla` y `Asignatura`) para que tenga el mismo formato que la hoja `BASE`.

**Si algo falla:** el mensaje indica cuál archivo o cuál hoja no se pudo leer. Lo más común es que el Excel no esté cargado en Colab o que el nombre del archivo o de la hoja sea distinto.

In [ ]:
try:
    cliente_ai = OpenAI(
        api_key=userdata.get('OPENAI_API_KEY'),
        base_url="https://api.groq.com/openai/v1"
    )
    modelo_elegido = "openai/gpt-oss-120b"
    print("[Exito] API key y modelo IA funcionando correctamente")
except Exception as e:
    print(f"[Error] Al conectar con la API: {e}")

ARCHIVO_EXCEL = 'HORARIOS_SIMULADOS_CORREGIDO.xlsx'

try:
    df_horarios = pd.read_excel(ARCHIVO_EXCEL, sheet_name='BASE')
    print("[Exito] Excel de horarios cargado correctamente.")
except Exception as e:
    print(f"[Error] Al cargar el Excel: {e}")
    df_horarios = pd.DataFrame()

try:
    df_competencias = pd.read_excel(ARCHIVO_EXCEL, sheet_name='OPTATIVOS COMPETENCIAS GLOBALES')
    df_competencias = df_competencias.rename(columns={"Sigla": "Sigla Asignatura", "Asignatura": "Nombre Asignatura"})
    print("[Exito] Hoja de Competencias Globales cargada correctamente.")
except Exception as e:
    print(f"[Aviso] No se pudo cargar la hoja de Competencias Globales: {e}")
    df_competencias = pd.DataFrame()

[Exito] API key y modelo IA funcionando correctamente
[Exito] Excel de horarios cargado correctamente.
[Exito] Hoja de Competencias Globales cargada correctamente.


# 3. PREPARACIÓN DEL EXCEL Y HERRAMIENTAS DE NORMALIZACIÓN

**¿Qué hace esta celda?**
Deja los datos listos para ser comparados. Convierte los horarios escritos como texto en números, y define herramientas para que textos escritos de distinta forma se puedan reconocer como iguales.

**Convertir horarios en números**
En el Excel un horario aparece como texto, por ejemplo `Lu 08:30:00 - 10:00:00`. Para que el computador pueda detectar choques, la celda lo separa en tres datos nuevos:
- **Dia:** `Lu`, `Ma`, `Mi`, `Ju`, `Vi` o `Sa`.
- **Inicio:** minutos transcurridos desde la medianoche (08:30 queda como 510).
- **Fin:** igual que el inicio, para la hora de término (10:00 queda como 600).

Si un horario viene con un formato raro o con la hora de término menor a la de inicio, se deja vacío en lugar de adivinar.

**Herramientas definidas**
| Nombre | Qué hace |
|---|---|
| `quitar_acentos` y `normalizar` | Pasan un texto a minúsculas, sin acentos y sin espacios de más, para que "Mecánica" y "mecanica" se consideren iguales. |
| `parsear_horario` | Convierte un horario escrito en el trío día, inicio y fin. |
| `preparar_horarios` | Aplica lo anterior a toda una tabla del Excel. |
| `CARRERAS_EXCEL` | Diccionario que indica cómo se llama cada carrera dentro del Excel. Varias menciones (Desarrollo, Ciencia de Datos, etc.) comparten la misma carrera de Informática. |
| `catalogo_siglas` | Lista de todas las siglas del Excel con su nombre. Sirve para comprobar si un código leído desde una imagen realmente existe. |

Al terminar se muestra cuántas filas del Excel se prepararon y cuántas tienen horario fijo.

In [ ]:
PATRON_HORARIO = re.compile(r"^(Lu|Ma|Mi|Ju|Vi|Sa|Do)\s+(\d{1,2}):(\d{2}):\d{2}\s*-\s*(\d{1,2}):(\d{2}):\d{2}$")
PATRON_SIGLA = re.compile(r"^[A-Z]{3}\d{4}$")
DIAS_SEMANA = ["Lu", "Ma", "Mi", "Ju", "Vi", "Sa"]

CARRERAS_EXCEL = {
    "Mecánica": ["ING. EN MECÁNICA AUTOM. Y AUTOTRÓNICA"],
    "Finanzas": ["ING. EN ADMINISTRACIÓN MENCIÓN FINANZAS", "INGENIERÍA EN ADMINIST. MENCIÓN FINANZAS"],
    "Automatización": ["INGENIERÍA EN INFORMÁTICA"],
    "Ciencia de Datos": ["INGENIERÍA EN INFORMÁTICA"],
    "Desarrollo": ["INGENIERÍA EN INFORMÁTICA"],
    "Ingeniería de Datos": ["INGENIERÍA EN INFORMÁTICA"],
    "Inteligencia Artificial": ["INGENIERÍA EN INFORMÁTICA"]
}

def quitar_acentos(texto):
    return "".join(c for c in unicodedata.normalize("NFD", str(texto)) if unicodedata.category(c) != "Mn")

def normalizar(texto):
    return re.sub(r"\s+", " ", quitar_acentos(texto)).strip().lower()

def parsear_horario(texto):
    coincidencia = PATRON_HORARIO.match(str(texto).strip())
    if not coincidencia:
        return (None, None, None)
    dia, h1, m1, h2, m2 = coincidencia.groups()
    inicio = int(h1) * 60 + int(m1)
    fin = int(h2) * 60 + int(m2)
    if fin <= inicio:
        return (None, None, None)
    return (dia, inicio, fin)

def preparar_horarios(df):
    """Agrega las columnas Dia, Inicio y Fin (en minutos) a una tabla de horarios."""
    if df.empty:
        return df
    tripletas = df["Horario"].map(parsear_horario)
    df["Dia"] = [t[0] for t in tripletas]
    df["Inicio"] = [t[1] for t in tripletas]
    df["Fin"] = [t[2] for t in tripletas]
    return df

df_horarios = preparar_horarios(df_horarios)
df_competencias = preparar_horarios(df_competencias)

catalogo_siglas = {}
if not df_horarios.empty:
    catalogo_siglas = df_horarios.drop_duplicates("Sigla Asignatura").set_index("Sigla Asignatura")["Nombre Asignatura"].to_dict()
    print(f"[Exito] {len(df_horarios)} filas del Excel preparadas, {df_horarios['Dia'].notna().sum()} con horario fijo.")

[Exito] 840 filas del Excel preparadas, 840 con horario fijo.


# 4. LECTURA DE LAS MALLAS

**¿Qué hace esta celda?**
Enseña al programa a leer una imagen de malla curricular. No ejecuta la lectura todavía: solo define las funciones que se usarán en el paso siguiente.

**¿Cómo se ve una malla?**
Es una imagen con 8 columnas (una por semestre). Cada columna tiene un encabezado con una barra amarilla y, debajo, tarjetas blancas: una por asignatura, con su código, nombre y categoría.

**El proceso, paso a paso**
1. **Detectar las columnas:** se busca la barra amarilla de cada encabezado para saber dónde empieza y termina cada semestre.
2. **Detectar las tarjetas:** se ubican los rectángulos blancos y se asigna cada uno a su semestre según su posición.
3. **Leer cada tarjeta por separado:** se aplica OCR al código, al nombre y a la categoría.
4. **Votar el código:** el código se lee varias veces con distintos ajustes de imagen y gana la lectura que más se repite. Esto reduce errores como confundir la letra O con el número 0.
5. **Reconocer los cupos electivos:** las tarjetas "CUPO ELECTIVO" no tienen código. Se guardan como cupos en el semestre donde comienzan.

**Principales funciones**
| Función | Qué hace |
|---|---|
| `detectar_columnas` | Encuentra las columnas de semestre por el color amarillo. |
| `detectar_tarjetas` | Encuentra las tarjetas blancas de la imagen. |
| `_ocr_codigo` | Lee el código de una tarjeta con varias pruebas y votación. |
| `_ocr_texto_tarjeta` | Lee todo el texto de una tarjeta, línea por línea. |
| `_separar_nombre` | Separa el nombre, la categoría, la duración y si es un cupo. |
| `corregir_sigla` | Marca si el código existe en el Excel. Solo corrige un carácter cuando la lectura fue dudosa. |
| `tipo_cupo` | Clasifica el cupo: Formación Cristiana, Competencias Globales u Optativo disciplinar. |
| `leer_malla` | Función principal: recibe la imagen y devuelve las asignaturas y cupos de cada semestre. |


In [ ]:
LANG_OCR = "spa"

PATRON_SIGLA_MALLA = re.compile(r"[A-Z]{3}\d{4}")
CATEGORIAS = {
    "especialidades": "Especialidad",
    "formacion sello": "Formación Sello",
    "formacion general": "Formación General",
    "formacion optativa": "Formación Optativa",
}


def _norm(t):
    return normalizar(t)



def detectar_columnas(bgr):
    """Las 8 columnas de semestre se detectan por la barra amarilla bajo cada encabezado."""
    b, g, r = bgr[:, :, 0].astype(int), bgr[:, :, 1].astype(int), bgr[:, :, 2].astype(int)
    amarillo = (r > 220) & (g > 150) & (g < 215) & (b < 90)
    alto_busqueda = min(bgr.shape[0], 1200)
    filas = np.where(amarillo[:alto_busqueda].sum(axis=1) > bgr.shape[1] * 0.4)[0]
    if len(filas) == 0:
        return [], 0
    y = int(np.median(filas))
    linea = amarillo[max(0, y - 2):y + 3].any(axis=0)
    xs = np.where(linea)[0]
    corridas, ini = [], xs[0]
    for a, b_ in zip(xs, xs[1:]):
        if b_ - a > 12:
            corridas.append((ini, a))
            ini = b_
    corridas.append((ini, xs[-1]))
    corridas = [c for c in corridas if c[1] - c[0] > bgr.shape[1] * 0.05]
    return corridas, int(filas.max())



def detectar_tarjetas(bgr, y_min):
    blanco = ((bgr[:, :, 0] >= 254) & (bgr[:, :, 1] >= 254) & (bgr[:, :, 2] >= 254)).astype(np.uint8) * 255
    blanco[:y_min + 5, :] = 0
    cerrado = cv2.morphologyEx(blanco, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)))
    contornos, _ = cv2.findContours(cerrado, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    tarjetas = []
    for c in contornos:
        x, y, w, h = cv2.boundingRect(c)
        if w >= bgr.shape[1] * 0.07 and h >= bgr.shape[0] * 0.025 and w * h > 40000:
            tarjetas.append((x, y, w, h))
    return sorted(tarjetas, key=lambda t: (t[1], t[0]))



A2D = {"O": "0", "Q": "0", "I": "1", "L": "1", "S": "5", "B": "8", "Z": "2", "G": "6"}
D2A = {"0": "O", "1": "I", "5": "S", "8": "B", "2": "Z", "6": "G"}


def _forzar_formato(s7):
    """3 letras + 4 digitos: corrige confusiones tipicas I/1, O/0, S/5, B/8, Z/2, G/6."""
    if len(s7) != 7:
        return None
    r = "".join((D2A.get(c, c) if i < 3 else A2D.get(c, c)) for i, c in enumerate(s7))
    return r if PATRON_SIGLA_MALLA.fullmatch(r) else None


def candidatos_codigo(txt):
    s = re.sub(r"[^A-Z0-9]", "", txt.upper())
    crudos = set()
    if len(s) == 7:
        crudos.add(s)
    elif len(s) == 8:
        crudos |= {s[:i] + s[i + 1:] for i in range(8)}
    elif len(s) > 8:
        crudos |= {s[i:i + 7] for i in range(len(s) - 6)}
    return [c for c in (_forzar_formato(x) for x in crudos) if c]


def _ocr_codigo(bgr, caja, catalogo=None):
    x, y, w, h = caja
    k = bgr.shape[1] / 2768.0
    banda = bgr[y + int(5 * k):y + int(52 * k), x + int(4 * k):x + int(w * 0.66)]
    gris = cv2.cvtColor(banda, cv2.COLOR_BGR2GRAY)
    votos = []
    for escala in (3, 4):
        g = cv2.resize(gris, None, fx=escala, fy=escala, interpolation=cv2.INTER_CUBIC)
        for umbral in (170, 200, 225):
            _, bn = cv2.threshold(g, umbral, 255, cv2.THRESH_BINARY)
            bn = cv2.copyMakeBorder(bn, 20, 20, 20, 20, cv2.BORDER_CONSTANT, value=255)
            txt = pytesseract.image_to_string(bn, lang=LANG_OCR, config="--psm 7")
            votos.extend(candidatos_codigo(txt))
    if not votos:
        return None, 0.0
    conteo = {v: votos.count(v) for v in set(votos)}
    mejor = max(conteo.values())
    empatados = [v for v, c in conteo.items() if c == mejor]
    ganador = next((v for v in empatados if catalogo and v in catalogo), empatados[0])
    return ganador, min(1.0, conteo[ganador] / 6.0)


def _ocr_texto_tarjeta(bgr, caja):
    x, y, w, h = caja
    recorte = bgr[y + 3:y + h - 3, x + 3:x + w - 3]
    gris = cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(gris, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    datos = pytesseract.image_to_data(g, lang=LANG_OCR, config="--psm 6", output_type=pytesseract.Output.DICT)
    palabras = []
    for i, t in enumerate(datos["text"]):
        t = t.strip()
        if t and float(datos["conf"][i]) > -1:
            palabras.append((t, datos["left"][i], datos["top"][i], datos["height"][i]))
    lineas = []
    for p in sorted(palabras, key=lambda p: (p[2], p[1])):
        if lineas and abs(p[2] - lineas[-1][0][2]) < max(12, p[3] * 0.6):
            lineas[-1].append(p)
        else:
            lineas.append([p])
    return [" ".join(q[0] for q in sorted(l, key=lambda q: q[1])) for l in lineas]


ETIQUETAS = ["especialidades", "formacion sello", "formacion general", "formacion optativa"]


def _es_etiqueta(n):
    """True si la linea es la etiqueta de categoria (con o sin '5 semestres'), tolerando ruido de OCR."""
    limpio = re.sub(r"\b\d\s*semestres?\b", "", n).strip()
    if not limpio:
        return None
    for et in ETIQUETAS:
        if difflib.SequenceMatcher(None, limpio, et).ratio() >= 0.82:
            return et
    return None


def _limpiar_ruido_inicial(linea):
    """Quita iconos mal leidos al inicio ('i', '§', '()', 'J'...)."""
    toks = linea.split()
    while len(toks) > 1 and not re.search(r"[A-Za-z]{3,}", toks[0]) and _norm(toks[0]) not in {"de", "en", "el", "la", "y", "a"}:
        toks.pop(0)
    return " ".join(toks)


def _limpiar_nombre(nombre):
    """Corrige numeros romanos mal leidos ('|', 'l') y quita caracteres que romperian tablas markdown."""
    nombre = re.sub(r"\|+", lambda m: "I" * len(m.group()), nombre)
    toks = nombre.split()
    if len(toks) > 1 and re.fullmatch(r"[Il]{1,3}", toks[-1]):
        toks[-1] = "I" * len(toks[-1])
    nombre = " ".join(toks)
    return re.sub(r"[^\w\s().,&/\-ÁÉÍÓÚÜÑáéíóúüñ]", "", nombre).strip()


def _separar_nombre(lineas):
    """Devuelve (nombre, categoria, semestres_span, es_cupo) a partir de las lineas OCR de una tarjeta."""
    lineas = [l.strip() for l in lineas if l.strip()]
    es_cupo, categoria, span = False, None, 1
    restantes = []
    for linea in lineas:
        n = _norm(linea)
        if n.startswith("cupo"):
            es_cupo = True
            resto = re.sub(r"^(cupo\s*electivo|cupo)\s*", "", n)
            if len(resto) < 3:
                continue
            linea = linea[len(linea) - len(resto):] if len(linea) >= len(resto) else linea
        restantes.append(linea)

    while restantes:
        n = _norm(restantes[-1])
        m = re.search(r"(\d)\s*semestres?", n)
        et = _es_etiqueta(n)
        if et:
            categoria = CATEGORIAS.get(et)
            if m:
                span = int(m.group(1))
            restantes.pop()
        elif m and len(n) < 14:
            span = int(m.group(1))
            restantes.pop()
        else:
            break

    if restantes and not es_cupo:
        primera = restantes[0].split()
        if primera and (candidatos_codigo(primera[0]) or (re.fullmatch(r"[A-Z0-9]{5,8}", primera[0]) and len(primera) <= 3)):
            restantes.pop(0)
    restantes = [_limpiar_ruido_inicial(l) for l in restantes]
    return _limpiar_nombre(" ".join(restantes)), categoria, span, es_cupo


def tipo_cupo(nombre):
    n = _norm(nombre)
    if "cristiana" in n:
        return "formacion_cristiana"
    if "competencias globales" in n or "capacidades" in n:
        return "competencias_globales"
    if "disciplinar" in n:
        return "optativo_disciplinar"
    return "otro"



def corregir_sigla(sigla, nombre, catalogo, confianza=1.0):
    """catalogo: {sigla: nombre_excel}.
    Una lectura estable (confianza >= 0.66) NUNCA se reemplaza: solo se marca si existe en el Excel.
    Solo si la lectura es inestable se intenta corregir 1 caracter, y exigiendo nombre casi identico."""
    if sigla is None:
        return None, False
    def sim(a, b):
        return difflib.SequenceMatcher(None, _norm(a), _norm(b)).ratio() if a and b else 0.0
    if confianza >= 0.66 or sigla in catalogo:
        return sigla, sigla in catalogo
    vecinos = [s for s in catalogo if len(s) == len(sigla) and sum(a != b for a, b in zip(s, sigla)) == 1]
    buenos = [s for s in vecinos if nombre and sim(nombre, catalogo[s]) >= 0.85]
    if len(buenos) == 1:
        return buenos[0], True
    return sigla, False

def leer_malla(ruta, catalogo=None):
    catalogo = catalogo or {}
    bgr = cv2.imdecode(np.fromfile(ruta, dtype=np.uint8), cv2.IMREAD_COLOR)
    columnas, y_hdr = detectar_columnas(bgr)
    if not columnas:
        return {"semestres": {}, "electivos": False, "advertencias": ["No se detectaron columnas de semestre"]}
    tarjetas = detectar_tarjetas(bgr, y_hdr)
    semestres, advertencias = {}, []
    for (x, y, w, h) in tarjetas:
        izq, der = x, x + w
        solapes = [max(0, min(der, c1) - max(izq, c0)) / (c1 - c0) for c0, c1 in columnas]
        cubiertas = [i for i, s in enumerate(solapes) if s > 0.35]
        if not cubiertas:
            continue
        inicio = cubiertas[0]
        lineas = _ocr_texto_tarjeta(bgr, (x, y, w, h))
        nombre, categoria, span, es_cupo = _separar_nombre(lineas)

        span = max(span, len(cubiertas))
        sigla, conf = (None, 0.0) if es_cupo else _ocr_codigo(bgr, (x, y, w, h), catalogo)
        if es_cupo or (sigla is None and any(_norm(l).startswith(("electivo", "optativo")) for l in lineas[:3])):
            if not nombre:
                advertencias.append(f"Tarjeta de cupo sin nombre legible en x={x}, y={y}")
                continue
            item = {"tipo": "cupo", "sigla": None, "nombre": nombre, "categoria": categoria,
                    "semestres_span": span, "clase_cupo": tipo_cupo(nombre), "verificada": True}
        else:
            sigla, ok = corregir_sigla(sigla, nombre, catalogo, conf)
            if sigla is None:
                advertencias.append(f"Codigo ilegible en la tarjeta '{nombre}' (semestre {inicio + 1}); se buscara por nombre")
            item = {"tipo": "asignatura", "sigla": sigla, "nombre": nombre, "categoria": categoria,
                    "semestres_span": span, "verificada": ok, "confianza_ocr": round(conf, 2)}
        semestres.setdefault(inicio + 1, []).append((y, x, item))
    vistos = {}
    for n, lst in semestres.items():
        for _, _, it in lst:
            if it["sigla"]:
                vistos.setdefault(it["sigla"], []).append(n)
    for sg, ns in vistos.items():
        if len(ns) > 1:
            advertencias.append(f"Codigo {sg} repetido en los semestres {ns}: probable error de lectura, revisar")
    resultado = {}
    for n in sorted(semestres):
        resultado[n] = [it for _, _, it in sorted(semestres[n], key=lambda t: (t[0], t[1]))]
    return {"semestres": resultado, "electivos": any(i["tipo"] == "cupo" for v in resultado.values() for i in v),
            "advertencias": advertencias, "n_columnas": len(columnas)}

# 5. PROCESAMIENTO DE LAS MALLAS

**¿Qué hace esta celda?**
Ejecuta la lectura de las siete mallas curriculares y muestra un resumen de lo que encontró en cada una. Es el control de calidad del proyecto: permite verificar que la lectura fue correcta antes de usar el asistente.

**Qué hay que tener listo**
`mallas_archivos` es un diccionario que une cada carrera con el nombre de su imagen. Las siete imágenes deben estar cargadas en Colab con exactamente esos nombres.

**Qué muestra por cada malla**
- Cantidad de asignaturas, de cupos electivos y de semestres detectados.
- Una línea por semestre con los códigos encontrados. Los cupos aparecen marcados como `[cupo]` y las asignaturas cuyo código no se pudo leer, como `[sin codigo]`.
- Mensajes `[Revisar]` cuando algo parece dudoso, por ejemplo un código repetido en dos semestres.

**Cómo usar este resultado**
Abre la imagen de cada malla y compara semestre por semestre con la lista impresa. Una malla normal tiene 8 columnas. Si falta algún código o hay un aviso `[Revisar]`, conviene corregirlo antes de continuar.

**Dónde queda guardado**
Todo el resultado se guarda en `base_mallas_estructura`, que es la información que usará el motor de horarios. Si una malla falla, el error se registra y las demás siguen procesándose.

In [ ]:
mallas_archivos = {
    "Mecánica": "MALLA_CURRICULAR-Ingeniería-en-Mecánica-Automotriz-y-Autotrónica-DuocUC-2026.jpg",
    "Finanzas": "MALLA_CURRICULAR-Ingeniería-en-Administración-mención-Finanzas-DuocUC-2026.jpg",
    "Automatización": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Automatización-DuocUC-2026.jpg",
    "Ciencia de Datos": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Ciencia-de-Datos-DuocUC-2026.jpg",
    "Desarrollo": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Desarrollo-de-Software-DuocUC-2026.jpg",
    "Ingeniería de Datos": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Ingeniería-de-Datos-DuocUC-2026.jpg",
    "Inteligencia Artificial": "MALLA_CURRICULAR-Ingeniería-en-Informática-Mención-Inteligencia-Artificial-DuocUC-2026.jpg"
}

base_mallas_estructura = {}
for clave, archivo in mallas_archivos.items():
    try:
        print(f"Leyendo la malla: {clave}...")
        base_mallas_estructura[clave] = leer_malla(archivo, catalogo_siglas)
        malla = base_mallas_estructura[clave]
        semestres = malla["semestres"]
        n_asig = sum(1 for v in semestres.values() for i in v if i["tipo"] == "asignatura")
        n_cupos = sum(1 for v in semestres.values() for i in v if i["tipo"] == "cupo")
        print(f"   {n_asig} asignaturas y {n_cupos} cupos electivos en {len(semestres)} semestres "
              f"({malla.get('n_columnas', 0)} columnas detectadas)")
        for numero, items in semestres.items():
            detalle = [i["sigla"] if i["tipo"] == "asignatura" and i["sigla"] else
                       ("[cupo] " + i["nombre"][:28] if i["tipo"] == "cupo" else "[sin codigo] " + i["nombre"][:28])
                       for i in items]
            print(f"   Semestre {numero}: {', '.join(detalle)}")
        for aviso in malla.get("advertencias", []):
            print(f"   [Revisar] {aviso}")
    except Exception as e:
        base_mallas_estructura[clave] = {"semestres": {}, "electivos": False, "advertencias": [str(e)]}
        print(f"[Error] Al procesar la malla {clave}: {e}")

print("\n[Exito] Mallas curriculares en imagen procesadas.")

Leyendo la malla: Mecánica...
   36 asignaturas y 6 cupos electivos en 8 semestres (8 columnas detectadas)
   Semestre 1: OMI1101, OMI1102, ATI1101, SIR1101, PLC1110, MAT1111
   Semestre 2: OMI1103, OMI1104, OMI1105, ATI1102, ATI1103, FCE1100
   Semestre 3: OMI1106, OMI1107, OMI1108, ATI1104, ATI1105, [cupo] ELECTIVO FORMACIÓN CRISTIANA, [cupo] ELECTIVO CAPACIDADES Y COMPE, [cupo] OPTATIVO FORMACIÓN DISCIPLIN
   Semestre 4: OMI1109, ATI1106, ATI1107, ATI1108, INV1504, EAI4710
   Semestre 5: OMI1111, PPM1101, CAL1101, [cupo] ELECTIVO CAPACIDADES Y COMPE
   Semestre 6: OMI1110, GOI1101, CAL1102, ELI1102, [cupo] OPTATIVO FORMACIÓN DISCIPLIN
   Semestre 7: OMI1112, GOI1102, CAI1103, EAI4705, [cupo] OPTATIVO FORMACIÓN DISCIPLIN
   Semestre 8: PPM1102, OMI1113
Leyendo la malla: Finanzas...
   41 asignaturas y 3 cupos electivos en 8 semestres (8 columnas detectadas)
   Semestre 1: ICA0121, GEA1101, GEA1108, GAA1101, PLC1110, MAT1111
   Semestre 2: GEA1102, PAA1101, GAA1902, FCE1100, PLC2110, 

# 6. MOTOR DE HORARIOS (DATOS VERIFICADOS)

**¿Qué hace esta celda?**
Es el cerebro del sistema. Toma las asignaturas de un semestre, busca sus secciones en el Excel y arma el mejor horario posible sin choques. Todo se calcula con datos reales.

**El proceso, paso a paso**
1. **Buscar** (`buscar_materias`): para cada elemento de la columna del semestre se busca su código en el Excel, respetando la jornada elegida.
2. **Armar** (`armar_horario`): se prueban combinaciones de secciones hasta encontrar la que ubica la mayor cantidad de asignaturas sin choques. Primero se ubican las asignaturas y después los cupos electivos, en los espacios que quedaron libres.
3. **Presentar** (`generar_propuesta`): se construyen las tablas y los avisos que verá el estudiante.

**Estado de cada asignatura**
| Estado | Significado |
|---|---|
| Con horario | Se encontró una sección que cabe sin chocar. |
| Solo en otra jornada | Existe en el Excel, pero no en la jornada pedida. |
| Sin secciones en el Excel | La malla la incluye, pero el Excel no la publica. |
| Choque de horario | Tiene secciones, pero todas se superponen con otras ya elegidas. |


**Funciones de apoyo**
| Función | Qué hace |
|---|---|
| `normalizar_jornada`, `normalizar_semestre`, `normalizar_carrera` | Convierten lo que entiende la IA en valores exactos (por ejemplo, "tercero" pasa a ser 3 e "IA" pasa a ser Inteligencia Artificial). |
| `construir_secciones` | Agrupa las filas del Excel en secciones con docente, modalidad y bloques de horario. |
| `opciones_de_cupo` | Busca las opciones disponibles para un cupo electivo. |
| `hay_choque` | Indica si dos listas de bloques horarios se superponen. |
| `construir_tabla_horario`, `construir_tabla_detalle`, `construir_tabla_malla` | Generan las tres tablas que se muestran al estudiante. |

In [ ]:
def normalizar_jornada(valor):
    texto = normalizar(valor or "")
    if texto.startswith("diurn"):
        return "Diurno"
    if texto.startswith("vesper"):
        return "Vespertino"
    return None

def normalizar_semestre(valor):
    if valor is None:
        return None
    texto = normalizar(valor)
    numero = re.search(r"\d+", texto)
    if numero:
        return int(numero.group())
    palabras = {"primer": 1, "segundo": 2, "tercer": 3, "cuarto": 4, "quinto": 5,
                "sexto": 6, "septimo": 7, "octavo": 8, "noveno": 9, "decimo": 10}
    for palabra, valor_numerico in palabras.items():
        if palabra in texto:
            return valor_numerico
    return None

def normalizar_carrera(valor):
    texto = normalizar(valor or "")
    if not texto:
        return None
    alias = {"ia": "Inteligencia Artificial", "software": "Desarrollo", "automotriz": "Mecánica"}
    for clave in mallas_archivos:
        if normalizar(clave) == texto:
            return clave
    for palabra, clave in alias.items():
        if re.search(rf"\b{palabra}\b", texto):
            return clave
    coincidencias = [c for c in mallas_archivos if normalizar(c) in texto]
    if not coincidencias and len(texto) >= 5:
        coincidencias = [c for c in mallas_archivos if texto in normalizar(c)]
    if len(coincidencias) == 1:
        return coincidencias[0]
    return None

def celda(texto):
    """Texto seguro para una celda de tabla markdown."""
    return str(texto).replace("|", "/").replace("\n", " ")

def formatear_hora(minutos):
    return f"{minutos // 60:02d}:{minutos % 60:02d}"

NOMBRE_CUPO = {
    "formacion_cristiana": "Formación Cristiana",
    "competencias_globales": "Capacidades y Competencias Globales",
    "optativo_disciplinar": "Optativo de Formación Disciplinar",
    "otro": "Cupo electivo",
}
PALABRAS_NO_DISCIPLINAR = ["ajedrez", "basquetbol", "voleibol", "futbol", "tenis de mesa", "natacion",
                           "handbol", "atletismo", "entrenamiento comunicacional"]

def construir_secciones(filas):
    secciones = []
    for codigo, grupo in filas.groupby("Sección", sort=True):
        bloques = sorted({(d, int(i), int(f)) for d, i, f in zip(grupo["Dia"], grupo["Inicio"], grupo["Fin"]) if pd.notna(d)})
        docentes = grupo["Docente"].dropna().astype(str).unique().tolist()
        virtual = grupo["ASIGNATURA VIRTUAL SINCRONICA"].notna().any()
        secciones.append({
            "codigo": codigo,
            "sigla": grupo["Sigla Asignatura"].iloc[0],
            "nombre": grupo["Nombre Asignatura"].iloc[0],
            "bloques": bloques,
            "docente": " / ".join(docentes) if docentes else "Sin docente asignado en el Excel",
            "modalidad": "Online sincrona" if virtual else "Presencial"
        })
    return secciones

def opciones_de_cupo(clase, carrera, jornada):
    """Devuelve (secciones_en_la_jornada, jornadas_donde_existe) para un cupo electivo, solo con datos del Excel."""
    nombres_carrera = CARRERAS_EXCEL[carrera]
    if clase == "formacion_cristiana":
        base = df_horarios[df_horarios["Carrera"].isin(nombres_carrera) &
                           df_horarios["Nivel"].astype(str).str.contains("Formación Cristiana", case=False)]
    elif clase == "competencias_globales":
        base = df_competencias
    elif clase == "optativo_disciplinar":
        base = df_horarios[df_horarios["Carrera"].isin(nombres_carrera) & (df_horarios["Nivel"].astype(str) == "Optativos")]
        base = base[~base["Nombre Asignatura"].map(normalizar).str.contains("|".join(PALABRAS_NO_DISCIPLINAR))]
    else:
        return [], []
    if base is None or base.empty:
        return [], []
    en_jornada = base[base["Jornada"].isin([jornada, "Diurno y Vespertino"])]
    return construir_secciones(en_jornada) if not en_jornada.empty else [], sorted(base["Jornada"].unique().tolist())

_cache_planes = {}

def planes_de_la_malla(carrera):
    """Planes del Excel a los que pertenece esta malla, deducidos de las asignaturas cuyo codigo SI esta en el Excel.
    Se usa para no aceptar equivalencias por nombre desde otro plan/programa distinto."""
    if carrera not in _cache_planes:
        semestres = base_mallas_estructura.get(carrera, {}).get("semestres", {})
        siglas = {i["sigla"] for v in semestres.values() for i in v if i["tipo"] == "asignatura" and i["sigla"]}
        base = df_horarios[df_horarios["Carrera"].isin(CARRERAS_EXCEL[carrera])]
        _cache_planes[carrera] = set(base[base["Sigla Asignatura"].isin(siglas)]["Plan"].unique())
    return _cache_planes[carrera]

def buscar_materias(carrera, semestre, jornada):
    """Devuelve TODOS los elementos de la columna del semestre en la malla (asignaturas y cupos),
    cada uno con su estado real frente al Excel. Nada se agrega que no este en la malla."""
    items = base_mallas_estructura.get(carrera, {}).get("semestres", {}).get(semestre, [])
    base = df_horarios[df_horarios["Carrera"].isin(CARRERAS_EXCEL[carrera])]
    nombres_excel = base["Nombre Asignatura"].map(normalizar)
    materias = []
    for indice, item in enumerate(items):
        if item["tipo"] == "cupo":
            secciones, jornadas = opciones_de_cupo(item["clase_cupo"], carrera, jornada)
            if secciones:
                estado = "cupo_con_opciones"
            elif jornadas:
                estado = "cupo_solo_otra_jornada"
            else:
                estado = "cupo_sin_opciones"
            materias.append({"id": f"CUPO-{indice}", "tipo": "cupo", "sigla": None, "sigla_excel": None,
                             "nombre": item["nombre"], "clase_cupo": item["clase_cupo"],
                             "semestres_span": item.get("semestres_span", 1),
                             "secciones": secciones, "estado": estado, "jornadas_disponibles": jornadas})
            continue

        sigla = item["sigla"]
        filas = base[base["Sigla Asignatura"] == sigla] if sigla else base.iloc[0:0]
        sigla_excel = sigla
        posible_equivalente = None
        if filas.empty and item["nombre"]:

            por_nombre = base[nombres_excel == normalizar(item["nombre"])]
            if por_nombre["Sigla Asignatura"].nunique() == 1:
                posible_equivalente = por_nombre["Sigla Asignatura"].iloc[0]
        nombre = filas["Nombre Asignatura"].iloc[0] if not filas.empty else (item["nombre"] or "Nombre no legible en la malla")
        materia = {"id": sigla or f"SIN-CODIGO-{indice}", "tipo": "asignatura", "sigla": sigla or "(sin codigo)",
                   "sigla_excel": sigla_excel or "(sin codigo)", "nombre": nombre, "secciones": [],
                   "estado": "sin_secciones_en_excel", "jornadas_disponibles": [], "posible_equivalente": posible_equivalente}
        if not filas.empty:
            filas_jornada = filas[filas["Jornada"].isin([jornada, "Diurno y Vespertino"])]
            if filas_jornada.empty:
                materia["estado"] = "solo_en_otra_jornada"
                materia["jornadas_disponibles"] = sorted(filas["Jornada"].unique().tolist())
            else:
                materia["estado"] = "disponible"
                materia["secciones"] = construir_secciones(filas_jornada)
        materias.append(materia)
    return materias

def hay_choque(bloques_a, bloques_b):
    for dia_a, ini_a, fin_a in bloques_a:
        for dia_b, ini_b, fin_b in bloques_b:
            if dia_a == dia_b and ini_a < fin_b and ini_b < fin_a:
                return True
    return False

def armar_horario(materias):
    """Ubica la mayor cantidad posible de ASIGNATURAS de la malla sin choques (busqueda exhaustiva),
    y despues intenta acomodar los CUPOS electivos en los espacios que quedan libres."""
    candidatas = sorted([m for m in materias if m["tipo"] == "asignatura" and m["secciones"]],
                        key=lambda m: len(m["secciones"]))
    mejor = {"cantidad": -1, "seleccion": {}}
    contador = [0]

    def explorar(i, elegidas, ocupados):
        contador[0] += 1
        if contador[0] > 200000:
            return
        if len(elegidas) + (len(candidatas) - i) <= mejor["cantidad"]:
            return
        if i == len(candidatas):
            mejor["cantidad"] = len(elegidas)
            mejor["seleccion"] = dict(elegidas)
            return
        materia = candidatas[i]
        for seccion in materia["secciones"]:
            if not hay_choque(seccion["bloques"], ocupados):
                elegidas[materia["id"]] = seccion
                explorar(i + 1, elegidas, ocupados + seccion["bloques"])
                del elegidas[materia["id"]]
        explorar(i + 1, elegidas, ocupados)

    explorar(0, {}, [])
    seleccion = dict(mejor["seleccion"])

    ocupados = [b for s in seleccion.values() for b in s["bloques"]]
    for cupo in [m for m in materias if m["tipo"] == "cupo" and m["secciones"]]:

        orden = sorted(cupo["secciones"], key=lambda s: (not s["bloques"], s["nombre"], s["codigo"]))
        for seccion in orden:
            if not hay_choque(seccion["bloques"], ocupados):
                seleccion[cupo["id"]] = seccion
                ocupados = ocupados + seccion["bloques"]
                break
    return seleccion

def construir_tabla_horario(seleccion):
    bloques = []
    for _, seccion in seleccion.items():
        for dia, ini, fin in seccion["bloques"]:
            bloques.append((dia, ini, fin, seccion["nombre"], seccion["codigo"]))
    if not bloques:
        return ""
    franjas = sorted({(ini, fin) for _, ini, fin, _, _ in bloques})
    celdas = {}
    for dia, ini, fin, nombre, codigo in bloques:
        celdas[(dia, ini, fin)] = f"{celda(nombre)}<br>{codigo}"
    lineas = [
        "| Modulo Horario | Lunes | Martes | Miercoles | Jueves | Viernes | Sabado |",
        "|---|---|---|---|---|---|---|"
    ]
    for ini, fin in franjas:
        fila = [f"{formatear_hora(ini)} - {formatear_hora(fin)}"]
        for dia in DIAS_SEMANA:
            fila.append(celdas.get((dia, ini, fin), ""))
        lineas.append("| " + " | ".join(fila) + " |")
    return "\n".join(lineas)

def construir_tabla_detalle(seleccion, materias):
    lineas = [
        "| Sigla | Asignatura | Seccion | Docente | Modalidad | Horario |",
        "|---|---|---|---|---|---|"
    ]
    for materia in materias:
        seccion = seleccion.get(materia["id"])
        if seccion:
            if seccion["bloques"]:
                horario = "<br>".join(f"{dia} {formatear_hora(i)} - {formatear_hora(f)}" for dia, i, f in seccion["bloques"])
            else:
                horario = "Sin horario fijo en el Excel"
            etiqueta = " (cupo electivo)" if materia["tipo"] == "cupo" else ""
            lineas.append(f"| {seccion['sigla']} | {celda(seccion['nombre'])}{etiqueta} | {seccion['codigo']} | {celda(seccion['docente'])} | {seccion['modalidad']} | {horario} |")
    return "\n".join(lineas)

def construir_tabla_malla(seleccion, materias, jornada):
    """Lista COMPLETA de lo que trae la malla en este semestre y que paso con cada elemento."""
    lineas = ["| Tipo | Codigo | Segun la malla | Estado en esta propuesta |", "|---|---|---|---|"]
    for m in materias:
        if m["tipo"] == "asignatura":
            if m["id"] in seleccion:
                estado = f"Con horario, seccion {seleccion[m['id']]['codigo']}"
            elif m["estado"] == "sin_secciones_en_excel":
                estado = "El Excel no publica secciones para esta asignatura"
            elif m["estado"] == "solo_en_otra_jornada":
                estado = f"Solo publicada en jornada {', '.join(m['jornadas_disponibles'])}"
            else:
                estado = "Tiene secciones, pero todas chocan con otras asignaturas"
            lineas.append(f"| Asignatura | {celda(m['sigla'])} | {celda(m['nombre'])} | {estado} |")
        else:
            clase = NOMBRE_CUPO.get(m["clase_cupo"], "Cupo electivo")
            extra = f" (cupo valido por {m['semestres_span']} semestres)" if m["semestres_span"] > 1 else ""
            if m["id"] in seleccion:
                s = seleccion[m["id"]]
                estado = f"Opcion sugerida del Excel: {s['sigla']} {s['nombre']}, seccion {s['codigo']}"
            elif m["estado"] == "cupo_con_opciones":
                estado = "Hay opciones en el Excel, pero todas chocan con el horario armado"
            elif m["estado"] == "cupo_solo_otra_jornada":
                estado = f"Solo hay opciones en jornada {', '.join(m['jornadas_disponibles'])}"
            else:
                estado = "El Excel no publica opciones para este cupo"
            lineas.append(f"| Cupo electivo | - | {celda(m['nombre'])}{extra} [{clase}] | {celda(estado)} |")
    return "\n".join(lineas)

def generar_propuesta(carrera, semestre, jornada):
    materias = buscar_materias(carrera, semestre, jornada)
    seleccion = armar_horario(materias)

    asignaturas = [m for m in materias if m["tipo"] == "asignatura"]
    cupos = [m for m in materias if m["tipo"] == "cupo"]
    asignadas = [m for m in asignaturas if m["id"] in seleccion]
    sin_excel = [m for m in asignaturas if m["estado"] == "sin_secciones_en_excel"]
    otra_jornada = [m for m in asignaturas if m["estado"] == "solo_en_otra_jornada"]
    con_choque = [m for m in asignaturas if m["estado"] == "disponible" and m["id"] not in seleccion]
    cupos_asignados = [m for m in cupos if m["id"] in seleccion]
    cupos_sin_asignar = [m for m in cupos if m["id"] not in seleccion]

    datos_verificados = {
        "carrera": carrera,
        "semestre": semestre,
        "jornada": jornada,
        "total_asignaturas_en_malla": len(asignaturas),
        "total_cupos_electivos_en_malla": len(cupos),
        "asignadas": [{"sigla": m["sigla"], "nombre": m["nombre"], "seccion": seleccion[m["id"]]["codigo"]} for m in asignadas],
        "cupos_electivos_con_opcion_sugerida": [{"cupo": m["nombre"], "sigla": seleccion[m["id"]]["sigla"], "seccion": seleccion[m["id"]]["codigo"]} for m in cupos_asignados],
        "cupos_electivos_sin_opcion": [{"cupo": m["nombre"]} for m in cupos_sin_asignar],
        "sin_secciones_en_excel": [{"sigla": m["sigla"], "nombre": m["nombre"]} for m in sin_excel],
        "solo_en_otra_jornada": [{"sigla": m["sigla"], "nombre": m["nombre"], "jornadas": m["jornadas_disponibles"]} for m in otra_jornada],
        "no_ubicadas_por_choque_de_horario": [{"sigla": m["sigla"], "nombre": m["nombre"]} for m in con_choque],
        "codigos_distintos_entre_malla_y_excel": [{"sigla_malla": m["sigla"], "sigla_excel": m["sigla_excel"]} for m in asignaturas if m["sigla_excel"] != m["sigla"] and m["estado"] != "sin_secciones_en_excel"]
    }

    observaciones = []
    for m in asignaturas:
        if m["sigla_excel"] != m["sigla"] and m["estado"] != "sin_secciones_en_excel":
            observaciones.append(f"- {m['sigla']} - {m['nombre']}: la malla la registra con el codigo {m['sigla']}, pero el Excel la publica como {m['sigla_excel']}. Se usaron las secciones del Excel porque el nombre coincide exactamente.")

    avisos = []
    for m in sin_excel:
        pista = f" El Excel publica una asignatura con el mismo nombre bajo el codigo {m['posible_equivalente']}; no se uso porque el codigo no coincide con la malla (consultalo con tu coordinacion)." if m.get("posible_equivalente") else ""
        avisos.append(f"- {m['sigla']} - {m['nombre']}: pertenece a la malla de este semestre, pero el Excel no tiene secciones publicadas para ella.{pista}")
    for m in otra_jornada:
        avisos.append(f"- {m['sigla']} - {m['nombre']}: no tiene secciones en jornada {jornada}; el Excel solo la publica en jornada {', '.join(m['jornadas_disponibles'])}.")
    for m in con_choque:
        avisos.append(f"- {m['sigla']} - {m['nombre']}: tiene secciones, pero todas chocan con otras asignaturas ya ubicadas.")
    for m in cupos_sin_asignar:
        if m["estado"] == "cupo_solo_otra_jornada":
            avisos.append(f"- Cupo electivo '{m['nombre']}': el Excel solo publica opciones en jornada {', '.join(m['jornadas_disponibles'])}, no en {jornada}.")
        elif m["estado"] == "cupo_con_opciones":
            avisos.append(f"- Cupo electivo '{m['nombre']}': hay opciones en el Excel, pero todas chocan con el horario armado.")
        else:
            avisos.append(f"- Cupo electivo '{m['nombre']}': el Excel no publica opciones para este cupo; revisalo al momento de inscribir.")

    siglas_validas = {m["sigla"] for m in asignaturas} | {m["sigla_excel"] for m in asignaturas}
    siglas_validas |= {s["sigla"] for m in cupos for s in m["secciones"]}

    return {
        "tabla_horario": construir_tabla_horario(seleccion),
        "tabla_detalle": construir_tabla_detalle(seleccion, materias),
        "tabla_malla": construir_tabla_malla(seleccion, materias, jornada),
        "avisos": avisos,
        "observaciones": observaciones,
        "datos_verificados": datos_verificados,
        "siglas_validas": siglas_validas
    }

# 7. SISTEMA MULTI-AGENTE CON MEMORIA DE ESTADO Y SEGUIMIENTO

**¿Qué hace esta celda?**
Une todas las piezas anteriores en un flujo de conversación. Aquí viven los dos agentes de inteligencia artificial y la función que coordina todo el proceso.

**Los dos agentes**
| Agente | Su trabajo | Lo que NO puede hacer |
|---|---|---|
| **Agente 1: Analista** | Lee el mensaje del estudiante y extrae tres datos: carrera, semestre y jornada. Responde únicamente en formato JSON. | No puede inventar datos que el estudiante no haya dicho. |
| **Agente 2: Redactor** | Escribe un resumen de máximo 4 oraciones a partir de los datos ya verificados. | No puede nombrar siglas, docentes ni horarios que no estén en los datos. |

Entre ambos agentes trabaja Python, que es quien arma todas las tablas con datos reales.

**Filtro de seguridad**
`validar_texto_agente` revisa el resumen escrito por el Agente 2. Si encuentra una sigla que no existe en los datos verificados, lo descarta y usa un resumen básico de respaldo. Así se evita que la IA muestre información inventada.

**Memoria de estado**
El sistema recuerda la carrera, el semestre y la jornada entre mensajes. Por eso el estudiante puede escribir primero "Soy de Finanzas, segundo semestre, diurno" y luego solo "¿y en vespertino?", sin repetir todo.

**Qué hace `procesar_solicitud`, paso a paso**
1. Si el mensaje está vacío, pide al estudiante que escriba su solicitud.
2. Llama al Agente 1 para actualizar la memoria.
3. Verifica que el semestre exista en la malla de esa carrera.
4. Si faltan datos, indica qué se registró y qué falta por informar.
5. Si están los tres datos, genera la propuesta y llama al Agente 2 para el resumen.
6. Entrega la respuesta completa con este orden: resumen, tabla de la malla, horario propuesto, detalle de secciones, avisos y observaciones.

In [ ]:
# Agente 1: Analista que lee el mensaje y actualiza la memoria de estado
def agente_analista(mensaje_usuario, estado_actual):
    prompt = f"""
    Eres un analizador de texto estricto.

    ESTADO ACTUAL EN MEMORIA:
    {json.dumps(estado_actual, ensure_ascii=False)}

    NUEVO MENSAJE DEL USUARIO:
    "{mensaje_usuario}"

    CARRERAS VALIDAS (usa exactamente uno de estos nombres o null):
    {json.dumps(list(mallas_archivos.keys()), ensure_ascii=False)}

    REGLAS:
    1. Busca referencias a la carrera, semestre y jornada en el nuevo mensaje.
    2. "carrera" debe ser exactamente uno de los nombres de CARRERAS VALIDAS. Si el usuario no menciona una carrera o mencion clara, usa el valor del ESTADO ACTUAL.
    3. "semestre" debe ser un numero entero (ej. tercer semestre = 3, 4to = 4).
    4. "jornada" debe ser exactamente "Diurno" o "Vespertino".
    5. Si el usuario pide cambiar un dato existente (ej. "y en vespertino?"), reemplazalo.
    6. Si el mensaje no cambia un dato, manten el valor que ya estaba en el ESTADO ACTUAL.
    7. Nunca inventes un dato que el usuario no haya dicho: si no hay informacion, deja null.
    8. Responde UNICAMENTE con un objeto JSON valido con las claves "carrera", "semestre" y "jornada". No uses texto adicional ni explicaciones.
    """
    try:
        respuesta = cliente_ai.chat.completions.create(
            model=modelo_elegido,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        contenido = respuesta.choices[0].message.content
        bloque_json = re.search(r"\{.*\}", contenido, re.DOTALL)
        crudo = json.loads(bloque_json.group())
    except Exception as e:
        print("Error en Agente Analista:", e)
        return estado_actual
    return {
        "carrera": normalizar_carrera(crudo.get("carrera")) or estado_actual.get("carrera"),
        "semestre": normalizar_semestre(crudo.get("semestre")) or estado_actual.get("semestre"),
        "jornada": normalizar_jornada(crudo.get("jornada")) or estado_actual.get("jornada")
    }

def validar_texto_agente(texto, siglas_validas):
    sin_tablas = "\n".join(l for l in texto.replace("```", "").splitlines() if not l.strip().startswith("|")).strip()
    for sigla in re.findall(r"[A-Z]{3}\d{4}", sin_tablas):
        if sigla not in siglas_validas:
            return None
    return sin_tablas or None

# Agente 2: Redactor que resume unicamente los datos verificados
def agente_redactor(propuesta):
    resumen_base = (f"Propuesta construida solo con la malla y el Excel de horarios para {propuesta['datos_verificados']['carrera']}, "
                    f"semestre {propuesta['datos_verificados']['semestre']}, jornada {propuesta['datos_verificados']['jornada']}.")
    prompt = f"""
    Eres el Asistente Virtual DUOC UC.
    Redacta un resumen breve de la propuesta de horario usando UNICAMENTE los datos verificados.

    DATOS VERIFICADOS:
    {json.dumps(propuesta['datos_verificados'], ensure_ascii=False)}

    REGLAS ESTRICTAS:
    1. Maximo 4 oraciones.
    2. Indica cuantas asignaturas del semestre quedaron con horario asignado de un total de las que tiene la malla.
    3. Si hay asignaturas sin secciones en el Excel, en otra jornada o con choque, mencionalas por su sigla tal como aparecen en los datos y sugiere consultar con la sede o coordinacion.
    4. Prohibido nombrar asignaturas, siglas, secciones, docentes u horarios que no esten en los datos verificados.
    5. No generes tablas. No utilices emojis.
    6. Si hay codigos distintos entre malla y Excel, indicalo en una sola oracion.
    7. Los cupos electivos (Formacion Cristiana, Competencias Globales, Optativos) no son asignaturas fijas: son cupos en los que el estudiante elige una opcion. Nombralos como cupos y di que las opciones indicadas son solo sugerencias tomadas del Excel.
    """
    try:
        respuesta = cliente_ai.chat.completions.create(
            model=modelo_elegido,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )
        texto = validar_texto_agente(respuesta.choices[0].message.content, propuesta["siglas_validas"])
        return texto if texto else resumen_base
    except Exception as e:
        print("Error en Agente 2:", e)
        return resumen_base

def procesar_solicitud(mensaje, estado_actual):
    if not mensaje.strip():
        return "Por favor, ingresa tu solicitud.", estado_actual

    nuevo_estado = agente_analista(mensaje, estado_actual)

    carrera = nuevo_estado.get("carrera")
    semestre = nuevo_estado.get("semestre")
    jornada = nuevo_estado.get("jornada")

    if carrera and semestre and semestre not in base_mallas_estructura.get(carrera, {}).get("semestres", {}):
        disponibles = sorted(base_mallas_estructura.get(carrera, {}).get("semestres", {}).keys())
        nuevo_estado["semestre"] = None
        msg = f"La malla de {carrera} no tiene un semestre {semestre} registrado."
        if disponibles:
            msg += f" Semestres disponibles: {', '.join(str(s) for s in disponibles)}."
        return msg + " Por favor, indicame un semestre valido.", nuevo_estado

    faltantes = []
    if not carrera: faltantes.append("tu carrera o mencion")
    if not semestre: faltantes.append("el semestre a cursar")
    if not jornada: faltantes.append("tu jornada (Diurna o Vespertina)")

    if faltantes:
        recibidos = []
        if carrera: recibidos.append(f"Carrera: {carrera}")
        if semestre: recibidos.append(f"Semestre: {semestre}")
        if jornada: recibidos.append(f"Jornada: {jornada}")

        msg = "Datos registrados en memoria hasta el momento:\n"
        if recibidos:
            for item in recibidos:
                msg += f"- {item}\n"
            msg += "\n"
        else:
            msg += "- Ningun dato registrado aun.\n\n"

        msg += f"Para proceder con la generacion del boletin, por favor indicame: {', '.join(faltantes)}."
        if not carrera:
            msg += f"\n\nCarreras disponibles: {', '.join(mallas_archivos.keys())}."
        return msg, nuevo_estado

    if df_horarios.empty:
        return "No se pudo cargar el Excel de horarios, por lo que no es posible generar la propuesta.", nuevo_estado

    propuesta = generar_propuesta(carrera, semestre, jornada)
    resumen = agente_redactor(propuesta)

    partes = [resumen]
    partes.append(f"### Asignaturas de la malla en el semestre {semestre}\n\n" + propuesta["tabla_malla"])
    if propuesta["tabla_horario"]:
        partes.append("### Horario propuesto\n\n" + propuesta["tabla_horario"])
        partes.append("### Detalle de secciones\n\n" + propuesta["tabla_detalle"])
    else:
        partes.append("No hay ninguna asignatura de este semestre con secciones publicadas en el Excel para la jornada indicada.")
    if propuesta["avisos"]:
        partes.append("### Asignaturas de la malla sin horario en esta propuesta\n\n" + "\n".join(propuesta["avisos"]))
    if propuesta["observaciones"]:
        partes.append("### Observaciones sobre los datos\n\n" + "\n".join(propuesta["observaciones"]))

    return "\n\n".join(partes), nuevo_estado

# 8. ESTILOS VISUALES (HTML Y CSS)

**¿Qué hace esta celda?**
Define la apariencia de la página web. Solo cambia el diseño: no interviene en cómo se calculan los horarios.

**Conceptos básicos**
- **HTML:** el lenguaje que define qué elementos tiene una página (títulos, textos, cajas).
- **CSS:** el lenguaje que define cómo se ven esos elementos (colores, tamaños, bordes).

**Qué contiene la celda**
- `estilos_css`: las reglas de diseño con los colores corporativos de Duoc UC: amarillo (`#FFB81C`), negro y blanco. Incluye el fondo gris claro de la página, el encabezado negro con borde amarillo y la clase `caja-negra`.
- `html_cabecera`: el bloque con el título "Portal Académico de Estudiantes" que se muestra arriba de la interfaz.

**La clase `caja-negra`**
Es el estilo de la zona donde aparece el boletín académico: fondo oscuro, texto blanco y borde amarillo. También define el formato de las tablas dentro de esa zona (encabezados amarillos sobre fondo negro y filas en gris oscuro).

**Para personalizar:** basta con cambiar los códigos de color al inicio de `estilos_css`; el cambio se reflejará en toda la página.

In [ ]:
estilos_css = """
:root {
    --duoc-yellow: #FFB81C;
    --duoc-black: #1A1A1A;
    --duoc-white: #FFFFFF;
}
body, .gradio-container {
    background-color: #EAECEF !important;
    font-family: 'Arial', sans-serif !important;
}
.gradio-container h1, .gradio-container h2, .gradio-container h3, .gradio-container p {
    color: var(--duoc-black) !important;
}
.encabezado-duoc {
    background-color: var(--duoc-black);
    padding: 25px;
    border-bottom: 5px solid var(--duoc-yellow);
    display: flex; align-items: center; justify-content: center;
    border-radius: 12px 12px 0 0;
}
.encabezado-duoc h1 { color: var(--duoc-yellow) !important; margin: 0; font-size: 28px; }
.encabezado-duoc p { color: #e0e0e0 !important; margin: 5px 0 0 0; }

.caja-negra {
    background-color: var(--duoc-black) !important;
    color: var(--duoc-white) !important;
    padding: 20px !important;
    border-radius: 10px !important;
    border: 2px solid var(--duoc-yellow) !important;
    box-shadow: 0px 4px 10px rgba(0,0,0,0.3);
}
.caja-negra h1, .caja-negra h2, .caja-negra h3, .caja-negra p, .caja-negra li, .caja-negra span {
    color: var(--duoc-white) !important;
}
.caja-negra table {
    background-color: #2D2D2D !important;
    color: var(--duoc-white) !important;
    border-color: #444 !important;
}
.caja-negra th {
    background-color: #000000 !important;
    color: var(--duoc-yellow) !important;
    text-align: center !important;
}
.caja-negra td {
    background-color: #2D2D2D !important;
    color: var(--duoc-white) !important;
    border-color: #444 !important;
    text-align: center !important;
}
.caja-negra td * { color: var(--duoc-white) !important; }
"""

html_cabecera = """
<div class="encabezado-duoc">
    <div style="text-align: center;">
        <h1>Portal Academico de Estudiantes</h1>
        <p>Buscador Inteligente de Horarios y Boletin Academico</p>
    </div>
</div>
"""

# 9. INTERFAZ GRÁFICA CON MEMORIA DE ESTADO

**¿Qué hace esta celda?**
Construye la página web final con **Gradio** y la conecta con todo lo anterior. Aquí se define lo que el estudiante ve y lo que ocurre cuando presiona el botón.

**Elementos de la página**
| Elemento | Función |
|---|---|
| Caja de texto | El estudiante escribe su solicitud en lenguaje natural, por ejemplo "Soy de Desarrollo, tercer semestre, jornada Diurna". |
| Botón **Enviar / Actualizar** | Ejecuta `procesar_solicitud`, que reúne todo el proceso. |
| Zona **Tu Boletín Académico** | Muestra el resumen, la tabla semanal, el detalle de secciones y los avisos. Usa el estilo `caja-negra`. |
| `gr.State` | La memoria de corto plazo de la sesión: guarda carrera, semestre y jornada entre mensajes. |

**Qué ocurre al presionar el botón**
1. El Agente 1 interpreta el mensaje y actualiza la memoria.
2. El motor de horarios arma la propuesta con los datos reales.
3. El Agente 2 redacta el resumen.
4. La respuesta aparece en la zona del boletín.

**Importante:** la memoria es individual y temporal. Cada persona que abre la página tiene la suya, y se pierde al recargar la página.

In [ ]:
with gr.Blocks(css=estilos_css) as portal_simulado:
    gr.HTML(html_cabecera)

    estado_memoria = gr.State({"carrera": None, "semestre": None, "jornada": None})

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Ingresa tu solicitud")
            entrada_usuario = gr.Textbox(
                label="",
                placeholder="Ej: Soy de Desarrollo, paso a 4to semestre en jornada Diurna.",
                lines=4,
                container=False
            )
            boton_generar = gr.Button("Enviar / Actualizar", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### Tu Boletin Academico")
            salida_bot = gr.Markdown(
                "La informacion de tu planificacion aparecera aqui.",
                elem_classes=["caja-negra"]
            )

    boton_generar.click(
        fn=procesar_solicitud,
        inputs=[entrada_usuario, estado_memoria],
        outputs=[salida_bot, estado_memoria]
    )

/tmp/ipykernel_3626/2208981604.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=estilos_css) as portal_simulado:


# 10. LANZAMIENTO DE LA APLICACIÓN

**¿Qué hace esta celda?**
Pone en marcha el portal y genera un enlace público temporal (`gradio.live`) que se puede abrir desde cualquier navegador o compartir para probar el prototipo.

**Cómo usarlo**
1. Ejecuta la celda y espera a que aparezca el enlace en la salida.
2. Abre el enlace en una pestaña nueva o compártelo con quien quiera probarlo.
3. Escribe una solicitud con carrera, semestre y jornada, y presiona **Enviar / Actualizar**.

**Qué debes saber**
- El enlace dura aproximadamente una semana y se desactiva al cerrar o reiniciar el entorno de Colab.
- La página solo funciona mientras el cuaderno siga en ejecución.
- Si el portal falla, revisa que todas las celdas anteriores se hayan ejecutado en orden y sin errores.

In [ ]:
print("Generando link del html en funcionamiento....")
portal_simulado.launch(share=True)

Generando link del html en funcionamiento....
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1c83e9ba10d74760b1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# PROMP DE PRUEBA
1. Hola, estudio Desarrollo de Software y paso a tercer semestre
2. diurno
3. como queda en vespertino